# Дообучение: Transformers, Datasets, PEFT и TRL
Илья Новицкий · Семинар · 28 ноября 2026 · Блок 4

Пройдём по всему стеку Hugging Face для дообучения на одной маленькой модели и одной простой задаче.
Цель — разобраться, **что делает каждая библиотека**; качество модели здесь вторично.

**Задача.** По команде голосовому помощнику вернуть JSON с интентом и слотами:

```
«разбуди меня в пять утра»  →  {"intent": "alarm_set", "slots": {"time": "пять утра"}}
```

| Шаг | Библиотека | Что делаем |
|---|---|---|
| 1 | Transformers | Загружаем модель и токенизатор, шаблон диалога, генерация |
| 2 | Datasets | Строим набор инструкций, Alpaca → сообщения, JSONL |
| 3 | — | Оцениваем исходную модель |
| 4 | PEFT | LoRA: какие слои, сколько параметров |
| 5 | TRL `SFTTrainer` | Обучение на инструкциях с потерями только на ответе |
| 6 | TRL `DPOTrainer` | Обучение по предпочтениям: выбранный и отклонённый ответ |
| 7 | PEFT | Сохранение, загрузка и слияние адаптера |
| 8 | bitsandbytes, Accelerate | QLoRA и запуск на нескольких GPU — обзор |

**Железо.** Модель `Qwen2.5-0.5B-Instruct`, 200 обучающих примеров. Весь ноутбук проходит на обычном
ноутбуке без видеокарты за 10–20 минут, на GPU или Apple Silicon — за несколько минут.

In [ ]:
# requirements (Python 3.10–3.12); в скобках — версии, на которых ноутбук прогнан:
#   torch>=2.4           (проверено 2.15.0)
#   transformers>=4.56   (проверено 5.22.2)
#   datasets>=3.0        (проверено 5.1.0)
#   peft>=0.17           (проверено 0.21.2)
#   trl>=0.24            (проверено 1.17.1)
#   accelerate>=1.0      (проверено 1.15.0)
#   pandas>=2.2          (проверено 3.1.1)
#   numpy>=1.26          (проверено 2.5.4)
#   matplotlib>=3.8      (проверено 3.11.2)
#   requests>=2.31       (проверено 2.34.2)
# Установка локально: pip install -r requirements.txt
# Google Colab: раскомментируйте строку ниже и перезапустите среду после установки.
# %pip install -q "torch>=2.4" "transformers>=4.56" "datasets>=3.0" "peft>=0.17" "trl>=0.24" "accelerate>=1.0" "pandas>=2.2" "numpy>=1.26" "matplotlib>=3.8" "requests>=2.31"
#
# bitsandbytes (для QLoRA) нужен только на GPU NVIDIA: pip install bitsandbytes

import sys
print("Python", sys.version.split()[0])

In [ ]:
import os
os.environ["TOKENIZERS_PARALLELISM"] = "false"
import re, json, random, hashlib, tarfile
from pathlib import Path
from time import perf_counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
import torch
from IPython.display import display

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"

if os.getenv("NLP_FORCE_CPU"):
    DEVICE = "cpu"
elif torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"
if DEVICE == "cpu":
    torch.set_num_threads(os.cpu_count() or 1)
# Веса держим в float32: модель маленькая, а bf16 на большинстве CPU работает очень медленно.
DTYPE = torch.float32
WORK_DIR = Path("seminar4_work"); WORK_DIR.mkdir(exist_ok=True)
CACHE_DIR = Path.home() / ".cache" / "nlp_course"; CACHE_DIR.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_colwidth", 120)
print("устройство:", DEVICE)

## 1. Transformers: модель, токенизатор и шаблон диалога
`AutoTokenizer` и `AutoModelForCausalLM` по названию на Hub скачивают нужные классы, веса и конфигурацию.
Инструктивная модель ждёт диалог в **своём** шаблоне — его применяет `apply_chat_template`.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = "left"          # для пакетной генерации: выравнивание слева
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE)
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} млн параметров, "
      f"{model.config.num_hidden_layers} слоёв, словарь {model.config.vocab_size}")

messages = [{"role": "system", "content": "Отвечай кратко."},
            {"role": "user", "content": "Что такое LoRA?"}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
print(prompt)

In [ ]:
@torch.no_grad()
def generate(model, list_of_messages, max_new_tokens=80):
    """Пакетная жадная генерация: список диалогов → список ответов."""
    model.eval()
    prompts = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=True) for m in list_of_messages]
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    out = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                         temperature=None, top_p=None, top_k=None, pad_token_id=tokenizer.pad_token_id)
    return tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print(generate(model, [messages])[0])

## 2. Datasets: набор инструкций
Берём русскоязычный MASSIVE: в нём слоты размечены прямо в тексте — `разбуди меня в [time : пять утра]`.
Оставляем четыре сценария (будильник, календарь, погода, транспорт) и примеры с простыми слотами.

In [ ]:
DATA_URL = "https://amazon-massive-nlu-dataset.s3.amazonaws.com/amazon-massive-dataset-1.1.tar.gz"
ARCHIVE_PATH = CACHE_DIR / "amazon-massive-dataset-1.1.tar.gz"
if not ARCHIVE_PATH.exists():
    ARCHIVE_PATH.write_bytes(requests.get(DATA_URL, timeout=300).content)
assert hashlib.sha256(ARCHIVE_PATH.read_bytes()).hexdigest() == "4cba5faa11c71437928e17cb1b9b3d8b8e727e7ea363a3a9a8045e19c0491577"
with tarfile.open(ARCHIVE_PATH, "r:gz") as archive:
    member = next(m for m in archive.getmembers() if m.name.endswith("/data/ru-RU.jsonl"))
    rows = [json.loads(l) for l in archive.extractfile(member).read().decode().splitlines()]

SLOT_TYPES = ["date", "time", "event_name", "place_name", "weather_descriptor", "transport_type", "timeofday", "person"]
SLOT_RE = re.compile(r"\[(\w+) : ([^\]]+)\]")
records = []
for r in rows:
    pairs = SLOT_RE.findall(r["annot_utt"])
    types = [t for t, _ in pairs]
    if r["scenario"] in {"alarm", "calendar", "weather", "transport"} and all(t in SLOT_TYPES for t in types) \
            and len(types) == len(set(types)):
        records.append({"partition": r["partition"], "text": r["utt"].strip(), "intent": r["intent"],
                        "slots": {t: v.strip() for t, v in pairs}})
data = pd.DataFrame(records).drop_duplicates("text")
INTENTS = sorted(data["intent"].unique())
train_df = data[data["partition"].eq("train")].sample(200, random_state=SEED)
test_df = data[data["partition"].eq("test")].sample(50, random_state=SEED)
print(len(INTENTS), "интентов:", INTENTS)
display(train_df.head(4))

Сначала сохраним данные в формате **Alpaca** — `instruction` / `input` / `output`, — а затем средствами
`datasets` превратим их в формат **сообщений**, который понимают шаблоны диалога и TRL.

In [ ]:
from datasets import load_dataset

INSTRUCTION = (
    "Разбери команду голосовому помощнику. Верни только JSON вида "
    '{"intent": "...", "slots": {"тип": "значение"}}. '
    "Допустимые intent: " + ", ".join(INTENTS) + ". "
    "Типы слотов: " + ", ".join(SLOT_TYPES) + ". Значение слота — фрагмент исходного текста."
)
def answer_json(row):
    return json.dumps({"intent": row["intent"], "slots": row["slots"]}, ensure_ascii=False)

for name, frame in [("train", train_df), ("test", test_df)]:
    with open(WORK_DIR / f"{name}_alpaca.jsonl", "w", encoding="utf-8") as f:
        for _, r in frame.iterrows():
            f.write(json.dumps({"instruction": INSTRUCTION, "input": r["text"], "output": answer_json(r)},
                               ensure_ascii=False) + "\n")

alpaca = load_dataset("json", data_files={"train": str(WORK_DIR / "train_alpaca.jsonl"),
                                          "test": str(WORK_DIR / "test_alpaca.jsonl")})
print(alpaca)
print(alpaca["train"][0])

In [ ]:
def alpaca_to_messages(example):
    return {"messages": [{"role": "system", "content": example["instruction"]},
                         {"role": "user", "content": example["input"]},
                         {"role": "assistant", "content": example["output"]}]}

chat_ds = alpaca.map(alpaca_to_messages, remove_columns=["instruction", "input", "output"])
print(chat_ds)
print(tokenizer.apply_chat_template(chat_ds["train"][0]["messages"], tokenize=False))

## 3. Оценка исходной модели
Метрики: доля валидного JSON, точность интента, F1 по слотам (пара «тип, значение» должна совпасть точно).

In [ ]:
def prompt_messages(text):
    return [{"role": "system", "content": INSTRUCTION}, {"role": "user", "content": text}]

def evaluate(model, frame=test_df, batch_size=25):
    outputs = []
    for i in range(0, len(frame), batch_size):
        outputs += generate(model, [prompt_messages(t) for t in frame["text"].iloc[i:i + batch_size]])
    valid = intent_ok = tp = fp = fn = 0
    for (_, row), out in zip(frame.iterrows(), outputs):
        try:
            pred = json.loads(out.strip())
            pred_slots = {(k, str(v)) for k, v in pred.get("slots", {}).items()}
            valid += 1
        except (json.JSONDecodeError, AttributeError):
            pred, pred_slots = {}, set()
        gold = set(row["slots"].items())
        intent_ok += pred.get("intent") == row["intent"]
        tp += len(gold & pred_slots); fp += len(pred_slots - gold); fn += len(gold - pred_slots)
    n = len(frame)
    return {"json_valid": valid / n, "intent_acc": intent_ok / n, "slot_f1": 2 * tp / max(1, 2 * tp + fp + fn)}, outputs

t0 = perf_counter()
scores = {"исходная": evaluate(model)[0]}
print(f"{perf_counter() - t0:.0f} с"); display(pd.DataFrame(scores).T.round(3))

## 4. PEFT: LoRA
`LoraConfig` описывает адаптер, `get_peft_model` встраивает его в модель. Посмотрим, в какие слои он
добавился и какую долю параметров мы будем обучать.

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(r=8, lora_alpha=16, lora_dropout=0.05, target_modules="all-linear", task_type="CAUSAL_LM")
preview = get_peft_model(AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE), lora_config)
preview.print_trainable_parameters()
lora_layers = sorted({name.split(".")[-2] for name, _ in preview.named_modules() if name.endswith("lora_A")})
print("Адаптированные матрицы в каждом блоке:", lora_layers)

q = preview.base_model.model.model.layers[0].self_attn.q_proj
d_in, d_out, r = q.in_features, q.out_features, lora_config.r
print(f"q_proj: полная матрица {d_in}×{d_out} = {d_in * d_out:,} параметров; "
      f"LoRA r={r}: {r} × ({d_in} + {d_out}) = {r * (d_in + d_out):,}")
del preview

## 5. TRL: обучение на инструкциях (`SFTTrainer`)
`SFTTrainer` сам применяет шаблон диалога к столбцу `messages`, встраивает LoRA по `peft_config` и
запускает цикл обучения из `transformers.Trainer`. `assistant_only_loss=True` оставляет в функции потерь
только ответ ассистента.

> **Важно для CPU.** В TRL по умолчанию `bf16=True`. На большинстве процессоров bf16 считается в десятки раз
> медленнее, поэтому ниже `bf16=False`. На GPU с поддержкой bf16 его можно включить.

In [ ]:
from trl import SFTConfig, SFTTrainer

sft_config = SFTConfig(
    output_dir=str(WORK_DIR / "sft"),
    num_train_epochs=2,
    learning_rate=2e-4,
    per_device_train_batch_size=8,
    lr_scheduler_type="cosine",
    max_length=256,
    assistant_only_loss=True,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=False, fp16=False,
    use_cpu=DEVICE == "cpu",
    seed=SEED,
)
sft_trainer = SFTTrainer(model=model, args=sft_config, train_dataset=chat_ds["train"],
                         processing_class=tokenizer, peft_config=lora_config)

example = sft_trainer.train_dataset[0]
labels = np.array(example["labels"]) if "labels" in example else None
if labels is not None:
    print(f"Токенов в примере: {len(labels)}, из них в потерях: {(labels != -100).sum()}")
    print("В потерях:", repr(tokenizer.decode(np.array(example["input_ids"])[labels != -100])))
else:   # в части версий TRL маска хранится отдельным полем
    print("Поля примера:", list(example.keys()))

In [ ]:
t0 = perf_counter()
sft_trainer.train()
print(f"SFT: {perf_counter() - t0:.0f} с")
log = pd.DataFrame(sft_trainer.state.log_history).dropna(subset=["loss"])
ax = log.plot(x="step", y="loss", figsize=(6, 3), legend=False, marker=".", title="Потери SFT")
ax.set_ylabel("loss"); plt.show()

sft_model = sft_trainer.model
scores["SFT (LoRA)"] = evaluate(sft_model)[0]
display(pd.DataFrame(scores).T.round(3))

## 6. TRL: обучение по предпочтениям (`DPOTrainer`)
Для DPO нужны тройки «запрос — выбранный ответ — отклонённый ответ». Соберём их сами: выбранный ответ —
эталонный JSON, отклонённый — JSON с **неверным интентом** из того же сценария. Так мы явно учим модель
различать похожие интенты.

Обучаем поверх SFT-модели: сначала **вливаем** SFT-адаптер в веса (`merge_and_unload`), затем добавляем
новый LoRA-адаптер для DPO. Исходной моделью для сравнения в DPO служит та же сеть с выключенным адаптером.

In [ ]:
from datasets import Dataset
from trl import DPOConfig, DPOTrainer

rng = random.Random(SEED)
def wrong_intent(intent):
    same_scenario = [i for i in INTENTS if i != intent and i.split("_")[0] == intent.split("_")[0]]
    return rng.choice(same_scenario or [i for i in INTENTS if i != intent])

pairs = Dataset.from_list([{
    "prompt": prompt_messages(r["text"]),
    "chosen": [{"role": "assistant", "content": answer_json(r)}],
    "rejected": [{"role": "assistant", "content": json.dumps({"intent": wrong_intent(r["intent"]), "slots": r["slots"]},
                                                             ensure_ascii=False)}],
} for _, r in train_df.head(120).iterrows()])
print(pairs[0]["chosen"][0]["content"], "  ≻  ", pairs[0]["rejected"][0]["content"])

merged_sft = sft_model.merge_and_unload()
dpo_config = DPOConfig(
    output_dir=str(WORK_DIR / "dpo"),
    beta=0.1,
    learning_rate=5e-5,
    num_train_epochs=1,
    per_device_train_batch_size=4,
    max_length=256,
    logging_steps=5,
    save_strategy="no",
    report_to="none",
    bf16=False, fp16=False,
    use_cpu=DEVICE == "cpu",
    seed=SEED,
)
dpo_trainer = DPOTrainer(model=merged_sft, args=dpo_config, train_dataset=pairs, processing_class=tokenizer,
                         peft_config=LoraConfig(r=8, lora_alpha=16, target_modules="all-linear", task_type="CAUSAL_LM"))
t0 = perf_counter()
dpo_trainer.train()
print(f"DPO: {perf_counter() - t0:.0f} с")

TRL логирует величины из функции потерь DPO:

* `rewards/chosen`, `rewards/rejected` — насколько (в единицах β·log-отношения к исходной модели) изменилась
  вероятность выбранного и отклонённого ответа;
* `rewards/margins` — их разность, должна расти;
* `rewards/accuracies` — доля пар, где выбранный ответ получил большую «награду».

In [ ]:
dpo_log = pd.DataFrame(dpo_trainer.state.log_history).dropna(subset=["loss"])
display(dpo_log[["step", "loss", "rewards/chosen", "rewards/rejected", "rewards/margins", "rewards/accuracies"]].round(3))
dpo_log.plot(x="step", y=["rewards/chosen", "rewards/rejected"], figsize=(6, 3), marker=".", title="Награды DPO")
plt.show()

scores["SFT + DPO"] = evaluate(dpo_trainer.model)[0]
display(pd.DataFrame(scores).T.round(3))

**Разберите результат.** Разрыв наград (`rewards/margins`) растёт, и почти все пары ранжируются верно,
но посмотрите на `rewards/chosen`: она уходит в минус. Модель «выигрывает» пары в основном за счёт того,
что сильнее понижает отклонённые ответы, — и вместе с ними теряет вероятность выбранных. Это признак
деградации из лекции 11, и на тесте он может проявиться падением точности интента при росте других метрик.

Что попробовать (упражнение): увеличить `beta` до 0,3–0,5, уменьшить `learning_rate` до 1e-5, взять меньше
шагов. Главное в механике: данные те же «запрос — ответ», но сигнал другой — не «вот правильный ответ»,
а «этот ответ лучше того»; поэтому после DPO качество обязательно перепроверяют на тесте.

## 7. PEFT: сохранение, загрузка и слияние

In [ ]:
from peft import PeftModel

adapter_dir = WORK_DIR / "dpo-adapter"
dpo_trainer.model.save_pretrained(adapter_dir)
size_mb = sum(f.stat().st_size for f in adapter_dir.rglob("*") if f.is_file()) / 2**20
print(sorted(p.name for p in adapter_dir.iterdir()), f"— {size_mb:.1f} МБ")

# Адаптер DPO обучен поверх SFT-модели, поэтому загружаем его на неё же
loaded = PeftModel.from_pretrained(merged_sft, adapter_dir)
final = loaded.merge_and_unload()
final.save_pretrained(WORK_DIR / "final-model")
tokenizer.save_pretrained(WORK_DIR / "final-model")
full_mb = sum(f.stat().st_size for f in (WORK_DIR / "final-model").rglob("*") if f.is_file()) / 2**20
print(f"Полная слитая модель: {full_mb:.0f} МБ против {size_mb:.1f} МБ адаптера")
print(generate(final, [prompt_messages("поставь будильник на семь утра в субботу")])[0])

## 8. QLoRA и Accelerate — обзор
**QLoRA** — та же LoRA, но базовая модель загружается в 4 битах через `bitsandbytes`. Работает на GPU NVIDIA;
ячейка ниже выполняется только там.

In [ ]:
if DEVICE == "cuda":
    from transformers import BitsAndBytesConfig
    from peft import prepare_model_for_kbit_training
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                             bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16)
    qmodel = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map="auto")
    qmodel = prepare_model_for_kbit_training(qmodel)
    print(f"Память под веса в 4 битах: {qmodel.get_memory_footprint() / 2**20:.0f} МБ")
    # дальше — тот же SFTTrainer(model=qmodel, ..., peft_config=lora_config)
else:
    print("GPU NVIDIA не найден: QLoRA пропускаем. Код ячейки показывает, как её включить.")

**Accelerate** — слой, на котором работают `Trainer` и TRL: он выбирает устройство, смешанную точность и
распределение по нескольким GPU. В ноутбуке он используется неявно. Тот же код обучения, сохранённый
в скрипт `train.py`, запускается на нескольких GPU без изменений:

```bash
accelerate config        # один раз: число GPU, точность, DeepSpeed/FSDP
accelerate launch train.py
```

## Итоги
| Библиотека | Что мы сделали |
|---|---|
| Transformers | Модель, токенизатор, шаблон диалога, генерация |
| Datasets | Alpaca в JSONL → `load_dataset("json")` → `map` в формат сообщений |
| PEFT | `LoraConfig`, `get_peft_model`, сохранение, `PeftModel.from_pretrained`, `merge_and_unload` |
| TRL | `SFTTrainer` с потерями на ответе, `DPOTrainer` на парах предпочтений |
| bitsandbytes, Accelerate | 4-битная загрузка для QLoRA, запуск обучения на нескольких GPU |

**Вопросы.** Почему потери считаются только по ответу? Чем данные для DPO отличаются от данных для SFT?
Что хранится в адаптере и почему он весит мегабайты? Зачем на CPU выключать bf16?

## Материалы
- [Документация TRL: SFTTrainer и DPOTrainer](https://huggingface.co/docs/trl)
- [Документация PEFT](https://huggingface.co/docs/peft)
- [Документация Datasets](https://huggingface.co/docs/datasets)
- [Hu et al. LoRA, 2021](https://arxiv.org/abs/2106.09685) · [Rafailov et al. DPO, 2023](https://arxiv.org/abs/2305.18290)